# Import Libraries

In [1]:
!pip install spacy kagglehub pandas matplotlib seaborn


In [2]:
!python -m spacy download en_core_web_lg
!python -m spacy download en_core_web_sm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 400.7/400.7 MB 2.6 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_lg')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 109.3 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [3]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import os
import warnings
warnings.filterwarnings('ignore')

import kagglehub
from collections import defaultdict

import spacy
from spacy import displacy
from spacy.pipeline import EntityRuler

# Load the small and medium English models
nlp_sm = spacy.load("en_core_web_sm")
nlp_lg = spacy.load("en_core_web_lg")

# Load and Prepare Dataset

In [4]:
data_path = kagglehub.dataset_download('alaakhaled/conll003-englishversion')
data_path

Using Colab cache for faster access to the 'conll003-englishversion' dataset.


'/kaggle/input/conll003-englishversion'

In [5]:
print(os.listdir(data_path))

['valid.txt', 'metadata', 'test.txt', 'train.txt']


In [6]:
file_path = os.path.join(data_path, 'train.txt')

lines = []
with open(file_path, 'r', encoding='utf-8') as f:
    lines = f.readlines()

print("First 20 lines of train.txt:")
print("".join(lines[:20]))

First 20 lines of train.txt:
-DOCSTART- -X- -X- O

EU NNP B-NP B-ORG
rejects VBZ B-VP O
German JJ B-NP B-MISC
call NN I-NP O
to TO B-VP O
boycott VB I-VP O
British JJ B-NP B-MISC
lamb NN I-NP O
. . O O

Peter NNP B-NP B-PER
Blackburn NNP I-NP I-PER

BRUSSELS NNP B-NP B-LOC
1996-08-22 CD I-NP O

The DT B-NP O
European NNP I-NP B-ORG



# Model-Based NER

In [12]:
import spacy
from collections import defaultdict

# ===== Load text from file =====
file_path = "/content/Ahmed_Fahim_test.txt"
with open(file_path, "r", encoding="utf-8") as f:
    text_data = f.read()

# ===== Load models =====
nlp_sm = spacy.load("en_core_web_sm")
nlp_lg = spacy.load("en_core_web_lg")
nlp_sm.max_length = 4000000
nlp_lg.max_length = 4000000

# ===== Function to get entities by label =====
def get_entities(doc):
    entity_dict = defaultdict(list)
    for ent in doc.ents:
        entity_dict[ent.label_].append(ent.text)
    # Remove duplicates
    for label in entity_dict:
        entity_dict[label] = list(set(entity_dict[label]))
    return entity_dict

# ===== Run SM Model =====
doc_sm = nlp_sm(text_data)
entities_sm = get_entities(doc_sm)

# ===== Run LG Model =====
doc_lg = nlp_lg(text_data)
entities_lg = get_entities(doc_lg)

# ===== Display SM Model Results =====
print("\n" + "="*40)
print("SM Model Entities")
print("="*40)
for label, items in entities_sm.items():
    print(f"{label} ({len(items)}): {sorted(items)}")

# ===== Display LG Model Results =====
print("\n" + "="*40)
print("LG Model Entities")
print("="*40)
for label, items in entities_lg.items():
    print(f"{label} ({len(items)}): {sorted(items)}")

# ===== Compare SM vs LG =====
all_labels = set(list(entities_sm.keys()) + list(entities_lg.keys()))

print("\n" + "="*50)
print("Comparison: SM vs LG")
print("="*50)

for label in all_labels:
    sm_set = set(entities_sm.get(label, []))
    lg_set = set(entities_lg.get(label, []))

    common = sm_set & lg_set
    only_sm = sm_set - lg_set
    only_lg = lg_set - sm_set

    print(f"\nLabel: {label}")
    print(f"  Common ({len(common)}): {sorted(common) if common else 'None'}")
    print(f"  Only SM ({len(only_sm)}): {sorted(only_sm) if only_sm else 'None'}")
    print(f"  Only LG ({len(only_lg)}): {sorted(only_lg) if only_lg else 'None'}")


========================================

SM Model Entities

========================================

PERSON (5): ['Ahmed', 'Ahmed Fahim', 'Deep Learning', 'Machine Learning', 'Mohamed Salah']

DATE (1): ['third-year']

...

Only LG (1): ['Reef Oasis']

Label: GPE

Common (3): ['Cairo', 'Egypt', 'Sharm El Sheikh']

Only SM (2): ['OpenAI', 'Python']

Only LG (0): None

SM Model: Sometimes classifies technical terms or projects as persons or locations → less accurate.

LG Model: More accurate, splits long phrases and distinguishes between persons, organizations, works of art, and locations.

Main difference: Accuracy in classification and level of detail; LG provides cleaner and more detailed results, while SM sometimes mixes categories.

# Rule-Based NER (EntityRuler)

In [13]:
import spacy
from collections import defaultdict

# ===== Load text from file =====
file_path = "/content/Ahmed_Fahim_test.txt"
with open(file_path, "r", encoding="utf-8") as f:
    text_data = f.read()

# ===== Load small model =====
nlp_rule = spacy.load("en_core_web_sm")
nlp_rule.max_length = 4000000

# ===== Add Entity Ruler BEFORE NER =====
ruler = nlp_rule.add_pipe("entity_ruler", before="ner")

# ===== Custom patterns =====
patterns = [
    {"label": "ORG", "pattern": "OpenAI"},
    {"label": "GPE", "pattern": "Cairo"},
    {"label": "PERSON", "pattern": "Mohamed Salah"}
]
ruler.add_patterns(patterns)

# ===== Apply pipeline =====
doc_rule = nlp_rule(text_data)

# ===== Organize entities by label =====
entity_dict = defaultdict(list)
for ent in doc_rule.ents:
    entity_dict[ent.label_].append(ent.text)

# Remove duplicates for clarity
for label in entity_dict:
    entity_dict[label] = list(set(entity_dict[label]))

# ===== Print results clearly =====
print("\n" + "="*40)
print("Rule-Based NER Entities")
print("="*40)
for label, items in entity_dict.items():
    print(f"{label} ({len(items)}): {sorted(items)}")


========================================

Rule-Based NER Entities

========================================

PERSON (5): ['Ahmed', 'Ahmed Fahim', 'Deep Learning', 'Machine Learning', 'Mohamed Salah']

DATE (1): ['third-year']

ORG (10): ['AI Developer and Data Scientist', 'Computer Vision', 'Data Science', 'OpenAI', 'Power BI', 'Reef 
Oasis', 'Riyadh\nUniversity', 'SQL', 'Sales Insights Analysis', 'the Faculty of Artificial Intelligence']

GPE (4): ['Cairo', 'Egypt', 'Python', 'Sharm El Sheikh']

In [5]:
# Increase the maximum length for the rule-based model
nlp_rule = spacy.load("en_core_web_sm")
nlp_rule.max_length = 4000000

ruler = nlp_rule.add_pipe("entity_ruler", before="ner")

# Add custom patterns
patterns = [
    {"label": "ORG", "pattern": "OpenAI"},
    {"label": "GPE", "pattern": "Cairo"},
    {"label": "PERSON", "pattern": "Mohamed Salah"}
]
ruler.add_patterns(patterns)

doc_rule = nlp_rule(text_data)
entities_rule = [(ent.text, ent.label_) for ent in doc_rule.ents]

# Highlight and Categorize Entities

In [14]:
entity_dict = defaultdict(list)

for ent in doc_lg.ents:
    entity_dict[ent.label_].append(ent.text)

for label, items in entity_dict.items():
    print(f"{label}: {set(items)}")

PERSON: {'Ahmed', 'Mohamed Salah', 'Ahmed Fahim'}

DATE: {'third-year'}

ORG: {'Instagram Reach Prediction', 'Data Scientist', 'Computer Vision', 'Python', 'SQL', 'Power BI', 'the Faculty 
of Artificial Intelligence', 'Riyadh\nUniversity', 'Machine Learning', 'Insights Analysis', 'Data Science', 'AI 
Developer'}

WORK_OF_ART: {'Deep Learning'}

LOC: {'Reef Oasis'}

GPE: {'Sharm El Sheikh', 'Cairo', 'Egypt'}

# Visualization with displacy

In [16]:
# Visualize entities from en_core_web_sm for a portion of the document
print("Visualizing entities from en_core_web_sm:")
displacy.render(doc_sm[:50], style='ent', jupyter=True)

# Visualize entities from en_core_web_lg for a portion of the document
print("\nVisualizing entities from en_core_web_lg:")
displacy.render(doc_lg[:50], style="ent", jupyter=True)

Visualizing entities from en_core_web_sm:

NameError: name 'displacy' is not defined

In [18]:
import spacy
from spacy import displacy

# ===== Assume doc_sm and doc_lg are already created =====
# doc_sm = nlp_sm(text_data)
# doc_lg = nlp_lg(text_data)

# ===== Visualize first 50 tokens from SM model =====
print("Visualizing entities from en_core_web_sm:")
displacy.render(doc_sm[:200], style='ent', jupyter=True)

# ===== Visualize first 50 tokens from LG model =====
print("\nVisualizing entities from en_core_web_lg:")
displacy.render(doc_lg[:200], style='ent', jupyter=True)


Visualizing entities from en_core_web_sm:

Visualizing entities from en_core_web_lg:

# Compare Results Between Models

In [20]:
import spacy
from collections import defaultdict

# ===== Load text from file =====
file_path = "/content/Ahmed_Fahim_test.txt"
with open(file_path, "r", encoding="utf-8") as f:
    text_data = f.read()

# ===== Load models =====
nlp_sm = spacy.load("en_core_web_sm")
nlp_lg = spacy.load("en_core_web_lg")
nlp_sm.max_length = 4000000
nlp_lg.max_length = 4000000

# ===== Rule-Based NER =====
nlp_rule = spacy.load("en_core_web_sm")
ruler = nlp_rule.add_pipe("entity_ruler", before="ner")
patterns = [
    {"label": "ORG", "pattern": "OpenAI"},
    {"label": "GPE", "pattern": "Cairo"},
    {"label": "PERSON", "pattern": "Mohamed Salah"}
]
ruler.add_patterns(patterns)
nlp_rule.max_length = 4000000

# ===== Function to get entities by label =====
def get_entities_by_label(doc):
    entity_dict = defaultdict(set)
    for ent in doc.ents:
        entity_dict[ent.label_].add(ent.text)
    return entity_dict

# ===== Run NER =====
doc_sm = nlp_sm(text_data)
entities_sm = get_entities_by_label(doc_sm)

doc_lg = nlp_lg(text_data)
entities_lg = get_entities_by_label(doc_lg)

doc_rule = nlp_rule(text_data)
entities_rule = get_entities_by_label(doc_rule)

# ===== Compute unique entities per model =====
all_labels = set(list(entities_sm.keys()) + list(entities_lg.keys()) + list(entities_rule.keys()))

print("\n" + "="*50)
print("Unique Entities per Model (excluding shared)")
print("="*50)

for label in all_labels:
    sm_set = entities_sm.get(label, set())
    lg_set = entities_lg.get(label, set())
    rule_set = entities_rule.get(label, set())

    # Remove shared entities
    shared_all = sm_set & lg_set & rule_set
    unique_sm = sm_set - lg_set - rule_set
    unique_lg = lg_set - sm_set - rule_set
    unique_rule = rule_set - sm_set - lg_set

    if unique_sm or unique_lg or unique_rule:
        print(f"\nLabel: {label}")
        if unique_sm:
            print(f"  Only SM ({len(unique_sm)}): {sorted(unique_sm)}")
        if unique_lg:
            print(f"  Only LG ({len(unique_lg)}): {sorted(unique_lg)}")
        if unique_rule:
            print(f"  Only Rule-Based ({len(unique_rule)}): {sorted(unique_rule)}")


==================================================

Unique Entities per Model (excluding shared)

==================================================

Label: ORG

Only LG (6): ['AI Developer', 'Data Scientist', 'Insights Analysis', 'Instagram Reach Prediction', 'Machine 
Learning', 'Python']

...

Only LG (1): ['Deep Learning']

Label: LOC

Only LG (1): ['Reef Oasis']

Label: GPE

Only SM (1): ['OpenAI']

In [21]:
import spacy
from collections import defaultdict

# ===== Load text from file =====
file_path = "/content/Ahmed_Fahim_test.txt"
with open(file_path, "r", encoding="utf-8") as f:
    text_data = f.read()

# ===== Load models =====
nlp_sm = spacy.load("en_core_web_sm")
nlp_lg = spacy.load("en_core_web_lg")
nlp_sm.max_length = 4000000
nlp_lg.max_length = 4000000

# ===== Rule-Based NER =====
nlp_rule = spacy.load("en_core_web_sm")
ruler = nlp_rule.add_pipe("entity_ruler", before="ner")
patterns = [
    {"label": "ORG", "pattern": "OpenAI"},
    {"label": "GPE", "pattern": "Cairo"},
    {"label": "PERSON", "pattern": "Mohamed Salah"}
]
ruler.add_patterns(patterns)
nlp_rule.max_length = 4000000

# ===== Function to get entities by label =====
def get_entities_by_label(doc):
    entity_dict = defaultdict(set)
    for ent in doc.ents:
        entity_dict[ent.label_].add(ent.text)
    return entity_dict

# ===== Run NER =====
doc_sm = nlp_sm(text_data)
entities_sm = get_entities_by_label(doc_sm)

doc_lg = nlp_lg(text_data)
entities_lg = get_entities_by_label(doc_lg)

doc_rule = nlp_rule(text_data)
entities_rule = get_entities_by_label(doc_rule)

# ===== Compare Results =====
all_labels = set(list(entities_sm.keys()) + list(entities_lg.keys()) + list(entities_rule.keys()))

print("\n" + "="*50)
print("Comparison of SM, LG, and Rule-Based NER")
print("="*50)

for label in all_labels:
    sm_set = entities_sm.get(label, set())
    lg_set = entities_lg.get(label, set())
    rule_set = entities_rule.get(label, set())

    # Shared and unique entities
    shared_all = sm_set & lg_set & rule_set
    shared_sm_lg = (sm_set & lg_set) - shared_all
    only_sm = sm_set - lg_set - rule_set
    only_lg = lg_set - sm_set - rule_set
    only_rule = rule_set - sm_set - lg_set

    print(f"\nLabel: {label}")
    print(f"  Shared by All ({len(shared_all)}): {sorted(shared_all) if shared_all else 'None'}")
    print(f"  Shared SM & LG only ({len(shared_sm_lg)}): {sorted(shared_sm_lg) if shared_sm_lg else 'None'}")
    print(f"  Only SM ({len(only_sm)}): {sorted(only_sm) if only_sm else 'None'}")
    print(f"  Only LG ({len(only_lg)}): {sorted(only_lg) if only_lg else 'None'}")
    print(f"  Only Rule-Based ({len(only_rule)}): {sorted(only_rule) if only_rule else 'None'}")


==================================================

Comparison of SM, LG, and Rule-Based NER

==================================================

Label: DATE

Shared by All (1): ['third-year']

...

Shared by All (3): ['Cairo', 'Egypt', 'Sharm El Sheikh']

Shared SM & LG only (0): None

Only SM (1): ['OpenAI']

Only LG (0): None

Only Rule-Based (0): None



### **1️⃣ PERSON**

* **Meaning:** Names of people or characters.
* **Purpose:** Detects any individual’s name mentioned in the text.

---

### **2️⃣ ORG (Organization)**

* **Meaning:** Organizations, companies, institutions, teams, or projects.
* **Purpose:** Captures official entities, companies, academic institutions, or large projects.

---

### **3️⃣ GPE (Geo-Political Entity)**

* **Meaning:** Places with political significance, like cities, countries, or official regions.
* **Purpose:** Detects names of cities, countries, capitals, or other politically recognized areas.
* **Note:** Sometimes it can misclassify non-geopolitical names.

---

### **4️⃣ WORK_OF_ART**

* **Meaning:** Artistic or creative works, including books, artworks, or projects considered “creative.”
* **Purpose:** Captures names of artistic, literary, or creative works.

---

### **5️⃣ LOC (Location)**

* **Meaning:** General or natural locations without political connotation.
* **Purpose:** Detects mountains, lakes, beaches, or any well-known place that is not a political entity.

---

### **6️⃣ DATE**

* **Meaning:** Any specific date, year, or time period.
* **Purpose:** Captures years, months, or any temporal description.

---

### 🔹 **Difference Between GPE and LOC**

| Label | Represents                                                   |
| ----- | ------------------------------------------------------------ |
| GPE   | Politically significant places (cities, countries, capitals) |
| LOC   | General or natural places without political significance     |

---


# Extracted Entities

In [24]:
import pandas as pd

# entities_lg is a list of tuples: (entity_text, entity_label)
# Convert it into a pandas DataFrame
df_entities = pd.DataFrame(entities_lg, columns=["Entity", "Label"])

# Optional: sort by Label and Entity for better readability
df_entities.sort_values(by=["Label", "Entity"], inplace=True)
df_entities.reset_index(drop=True, inplace=True)

# Display a sample of the extracted entities
print("Sample of extracted entities:")
print(df_entities.head(10))

# Save the DataFrame to a CSV file
output_csv = "extracted_entities.csv"
df_entities.to_csv(output_csv, index=False, encoding="utf-8")

print(f"\n✅ Entities successfully saved to '{output_csv}'")


Sample of extracted entities:

Empty DataFrame
Columns: [Entity, Label]
Index: []

✅ Entities successfully saved to 'extracted_entities.csv'

<h1>Advanced NER Comparator GUI</h1>

In [26]:
import gradio as gr
import spacy
from spacy.pipeline import EntityRuler
import pandas as pd
from collections import defaultdict

# ==========================================
# 1. Model Setup (Load Once)
# ==========================================
print("Loading Models... Please wait.")
nlp_sm = spacy.load("en_core_web_sm")
nlp_lg = spacy.load("en_core_web_lg")

# Setup Rule-Based Model
nlp_rule = spacy.load("en_core_web_sm")
ruler = nlp_rule.add_pipe("entity_ruler", before="ner")

# Define Custom Patterns
patterns = [
    {"label": "ORG", "pattern": "OpenAI"},
    {"label": "GPE", "pattern": "Cairo"},
    {"label": "PERSON", "pattern": "Mohamed Salah"},
    {"label": "PERSON", "pattern": "Ahmed Fahim"},
    {"label": "ORG", "pattern": "Riyadh University"},
    {"label": "GPE", "pattern": "Saudi Arabia"}
]
ruler.add_patterns(patterns)

# Increase max length for large texts
for nlp in [nlp_sm, nlp_lg, nlp_rule]:
    nlp.max_length = 4000000

# ==========================================
# 2. Helper Functions
# ==========================================

def get_gradio_entities(doc):
    """
    Format Spacy output for Gradio's HighlightedText component.
    Returns a list of (word, label) tuples.
    """
    entities = []
    last_idx = 0
    for ent in doc.ents:
        # Text before the entity
        if ent.start_char > last_idx:
            entities.append((doc.text[last_idx:ent.start_char], None))
        # The entity itself
        entities.append((ent.text, ent.label_))
        last_idx = ent.end_char
    # Remaining text
    if last_idx < len(doc.text):
        entities.append((doc.text[last_idx:], None))
    return entities

def get_entities_by_label(doc):
    """
    Extract unique entities grouped by label for set comparison.
    Returns: dict {label: set(words)}
    """
    entity_dict = defaultdict(set)
    for ent in doc.ents:
        entity_dict[ent.label_].add(ent.text)
    return entity_dict

def process_ner(text):
    """
    Main processing function:
    1. Runs all 3 models.
    2. Prepares visual output.
    3. Calculates logic intersection/difference for the table.
    """
    if not text.strip():
        return [], [], [], pd.DataFrame()

    # Run the models
    doc_sm = nlp_sm(text)
    doc_lg = nlp_lg(text)
    doc_rule = nlp_rule(text)

    # 1. Prepare Visual Outputs (Highlighted Text)
    output_sm = get_gradio_entities(doc_sm)
    output_lg = get_gradio_entities(doc_lg)
    output_rule = get_gradio_entities(doc_rule)

    # 2. Prepare Logic Comparison (Set Operations)
    entities_sm = get_entities_by_label(doc_sm)
    entities_lg = get_entities_by_label(doc_lg)
    entities_rule = get_entities_by_label(doc_rule)

    # Get all unique labels found across all models
    all_labels = set(list(entities_sm.keys()) + list(entities_lg.keys()) + list(entities_rule.keys()))

    comparison_data = []

    for label in all_labels:
        sm_set = entities_sm.get(label, set())
        lg_set = entities_lg.get(label, set())
        rule_set = entities_rule.get(label, set())

        # Get all unique words for this label
        all_words = sm_set | lg_set | rule_set

        for word in all_words:
            # Determine which models found this entity
            found_in = []
            if word in sm_set: found_in.append("SM")
            if word in lg_set: found_in.append("LG")
            if word in rule_set: found_in.append("Rule")

            # Determine Intersection Status
            status = ""
            if len(found_in) == 3:
                status = "Shared by All"
            elif "SM" in found_in and "LG" in found_in:
                status = "Shared SM & LG"
            elif len(found_in) == 1:
                status = f"Only {found_in[0]}"
            else:
                status = " & ".join(found_in)

            comparison_data.append({
                "Entity": word,
                "Label": label,
                "Found By": ", ".join(found_in),
                "Status": status
            })

    # Create and sort DataFrame
    df = pd.DataFrame(comparison_data)
    if not df.empty:
        # Sort by Label, then Status (to group shared/unique items together)
        df = df.sort_values(by=["Label", "Status", "Entity"])

    return output_sm, output_lg, output_rule, df

# ==========================================
# 3. Build the Professional GUI
# ==========================================
with gr.Blocks(theme=gr.themes.Soft()) as demo:

    # Centered Title without Emoji
    gr.HTML("""
        <div style="text-align: center;">
            <h1>Advanced NER Comparator</h1>
            <h3>Compare entities between Small, Large, and Rule-Based models using Set Operations.</h3>
        </div>
    """)

    with gr.Row():
        # Input Column
        with gr.Column(scale=1):
            input_text = gr.Textbox(
                lines=8,
                label="Input Text",
                placeholder="Paste text here...",
                value="OpenAI announced new models in San Francisco. Cairo is a beautiful city where Mohamed Salah lives."
            )

            # Buttons Row (Analyze + Restart)
            with gr.Row():
                btn_process = gr.Button("🔍 Analyze & Compare", variant="primary")
                # The Restart button clears all inputs and outputs
                btn_reset = gr.ClearButton(
                    components=[input_text],
                    value="🔄 Restart / Clear",
                    variant="secondary"
                )

            gr.Markdown("### Examples:")
            gr.Examples(
                examples=[
                    ["Apple is looking at buying U.K. startup for $1 billion. Mohamed Salah plays in Liverpool."],
                    ["Riyadh University is located in Saudi Arabia."]
                ],
                inputs=input_text
            )

        # Output Column
        with gr.Column(scale=2):
            with gr.Tabs():
                with gr.TabItem("🟢 Small Model"):
                    out_sm = gr.HighlightedText(label="SM Model Output", combine_adjacent=True)

                with gr.TabItem("🔵 Large Model"):
                    out_lg = gr.HighlightedText(label="LG Model Output", combine_adjacent=True)

                with gr.TabItem("🟠 Rule-Based"):
                    out_rule = gr.HighlightedText(label="Rule-Based Output", combine_adjacent=True)

                with gr.TabItem("📊 Detailed Comparison Table"):
                    gr.Markdown("### Intersection & Difference Analysis")
                    out_table = gr.Dataframe(
                        label="Entity Comparison",
                        headers=["Entity", "Label", "Found By", "Status"],
                        interactive=False
                    )

            # Add outputs to the clear button functionality
            btn_reset.add(out_sm)
            btn_reset.add(out_lg)
            btn_reset.add(out_rule)
            btn_reset.add(out_table)

    # Connect Logic to UI
    btn_process.click(
        fn=process_ner,
        inputs=input_text,
        outputs=[out_sm, out_lg, out_rule, out_table]
    )

# Launch App
demo.launch()

Loading Models... Please wait.

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://9791d161e35becdab4.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
